# Lab 0: Setup-Check

Dieses Notebook prüft in etwa einer Minute, ob Ihr Rechner für alle Labs bereit ist.
Führen Sie es einmal vollständig aus (`Run All`). Es verändert nichts an Ihrer Umgebung:
Der MLflow-Test schreibt in einen temporären Ordner, der am Ende wieder gelöscht wird.

Geprüft wird:

1. Python-Version und Paketversionen
2. Datensatz Banking77 und die vier Modelle im Lab-Ordner (wie `python prefetch.py --check`)
3. Ein Embedding-Modell läuft ohne Netzzugriff auf der CPU
4. MLflow kann einen Run mit Parameter, Metrik und Artefakt in eine SQLite-Datenbank schreiben

Die letzte Zelle meldet **Alles bereit** oder listet die Mängel mit der passenden Abhilfe auf.

In [ ]:
# Ordner labs/ finden und zum Arbeitsordner machen: Das Paket common und die Daten liegen dort.
# Nötig, wenn der Kernel woanders startet (PyCharm, VS Code mit Arbeitsordner im Pack-Root, Quelle in labs/_src/).
import os, sys
from pathlib import Path
for _p in [Path.cwd(), Path.cwd() / "labs", *Path.cwd().parents]:
    if (_p / "common" / "__init__.py").exists():
        os.chdir(_p)
        sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError("Ordner labs/ nicht gefunden. Öffnen Sie das Notebook aus dem Ordner labs/ des Kursmaterials.")

import importlib.metadata as metadata
import logging
import os
import shutil
import sys
import tempfile
import time
from pathlib import Path

problems: list[str] = []  # sammelt alle Mängel, damit ein Lauf die vollständige Liste liefert


def report(ok: bool, label: str, fix: str = "") -> bool:
    """Schreibt eine Prüfzeile und merkt sich den Mangel samt Abhilfe."""
    print(f"{'ok    ' if ok else 'MANGEL'} {label}")
    if not ok:
        problems.append(f"{label}. Abhilfe: {fix}" if fix else label)
    return ok

## 1. Python und Pakete

Die Labs sind mit Python 3.11 und den unten genannten Mindestständen getestet.
Drei Einträge haben zusätzlich eine Obergrenze: `mlflow` bleibt unter Version 3,
`sqlalchemy` unter 2.1 (MLflow 2.22 kann mit neueren Fassungen keine SQLite-Datenbank öffnen)
und `evidently` ist fest auf 0.4.40 gesetzt (Lab 4 nutzt die API dieser Version).

In [ ]:
from packaging.specifiers import SpecifierSet
from packaging.version import Version

REQUIRED = {
    "torch": ">=2.14",
    "transformers": ">=4.57",
    "sentence-transformers": ">=5.7",
    "mlflow": ">=2.22,<3",
    "sqlalchemy": ">=2.0,<2.1",
    "evidently": "==0.4.40",
    "fastapi": ">=0.142",
    "pydantic": ">=2.13",
    "optuna": ">=5.0",
    "onnx": ">=1.23",
    "onnxruntime": ">=1.30",
    "optimum": ">=2.1",
    "scikit-learn": ">=1.9",
    "numpy": ">=2.0",
    "pandas": ">=2.3",
    "fairlearn": ">=0.14",
    "matplotlib": ">=3.8",
    "ipywidgets": ">=8.1",
}

report(
    sys.version_info[:2] >= (3, 11),
    f"Python {sys.version.split()[0]} (benötigt: 3.11 oder neuer)",
    "virtuelle Umgebung mit Python 3.11 neu anlegen",
)
for package, spec in REQUIRED.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        report(False, f"{package} fehlt (benötigt: {spec})", "pip install -r requirements.txt")
        continue
    report(
        Version(installed) in SpecifierSet(spec),
        f"{package} {installed} (benötigt: {spec})",
        f'pip install "{package}{spec}"',
    )

## 2. Daten und Modelle

Die Labs laufen ohne Netzzugriff. Datensatz und Modelle müssen deshalb im Lab-Ordner liegen.
Der Daten-Hash stellt sicher, dass alle im Raum mit denselben Anfragen arbeiten.

In [ ]:
from common.paths import DATA_DIR, MODELS_DIR, offline

offline()

from common.models import CLASSIFIER_BASE, EMBEDDING_MODELS, SHARED_FOLDER

EXPECTED_HASH = "3176e3ac9aef7185"
PREFETCH = "einmal mit Netzzugang `python prefetch.py` im Ordner labs/ ausführen"

for name in ("train.parquet", "test.parquet", "intents.json"):
    report((DATA_DIR / name).exists(), f"data/banking77/{name}", PREFETCH)

# Kurznamen mit gemeinsamem Ordner (distilbert-raw und distilbert) nur einmal prüfen
folders = sorted({SHARED_FOLDER.get(name, name) for name in {**EMBEDDING_MODELS, **CLASSIFIER_BASE}})
for folder in folders:
    report((MODELS_DIR / folder / "config.json").exists(), f"models/{folder}", PREFETCH)

try:
    from common.data import dataset_hash, load_banking77

    train, test, intents = load_banking77()
    report(
        (len(train), len(test), len(intents)) == (9993, 3076, 77),
        f"Banking77: {len(train)} Trainings- und {len(test)} Testanfragen, {len(intents)} Anliegen",
        PREFETCH,
    )
    data_hash = dataset_hash(train, test)[:16]
    report(data_hash == EXPECTED_HASH, f"Daten-Hash {data_hash} (erwartet: {EXPECTED_HASH})", PREFETCH)
except Exception as exc:  # die Prüfung soll bis zum Ende laufen und alle Mängel nennen
    report(False, f"Datensatz lässt sich nicht laden ({type(exc).__name__}: {exc})", PREFETCH)

## 3. Ein Satz wird eingebettet

Das kleinste Embedding-Modell des Kurses (`minilm-l6`) übersetzt einen Satz in einen Vektor
mit 384 Dimensionen. Zusätzlich misst die Zelle, wie viele Texte je Sekunde Ihr Rechner schafft.
Der Wert hilft später beim Einordnen der Laufzeiten: Auf dem Referenzrechner (Apple M3 Max, CPU)
sind es rund 2 600 Texte je Sekunde, auf üblichen Laptops etwa drei- bis viermal weniger.

In [ ]:
texts_per_second = 0.0
try:
    import numpy as np

    from common.models import encode, load_encoder, timed_encode

    encoder = load_encoder("minilm-l6", device="cpu")
    vector = encode(encoder, ["I am still waiting on my card?"])[0]
    report(
        vector.shape == (384,) and abs(float(np.linalg.norm(vector)) - 1.0) < 1e-3,
        f"Embedding mit {vector.shape[0]} Dimensionen, Länge {np.linalg.norm(vector):.3f}",
        "Ordner labs/models/minilm-l6 löschen und `python prefetch.py` erneut ausführen",
    )
    sample = train["text"].head(512).tolist()
    encode(encoder, sample[:64])  # Warmlauf, damit die Messung nicht den Kaltstart enthält
    _, seconds = timed_encode(encoder, sample)
    texts_per_second = len(sample) / seconds
    print(f"       Durchsatz auf diesem Rechner: {texts_per_second:,.0f} Texte je Sekunde".replace(",", " "))
except Exception as exc:
    report(False, f"Embedding-Modell läuft nicht ({type(exc).__name__}: {exc})", PREFETCH)

## 4. MLflow schreibt einen Run

Der Test legt eine SQLite-Datenbank in einem temporären Ordner an, schreibt einen Run mit
Parameter, Metrik und Artefakt und liest ihn zurück. Ihre spätere Lab-Datenbank `labs/mlflow.db`
bleibt davon unberührt.

In [ ]:
tmp = Path(tempfile.mkdtemp(prefix="mlops_setup_"))
saved_env = {key: os.environ.get(key) for key in ("MLFLOW_TRACKING_URI", "MLFLOW_ARTIFACT_ROOT")}
try:
    os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{(tmp / 'mlflow.db').as_posix()}"
    os.environ["MLFLOW_ARTIFACT_ROOT"] = (tmp / "mlartifacts").as_posix()

    import mlflow

    from common.tracking import setup_mlflow

    logging.disable(logging.INFO)  # das Anlegen der Datenbank schreibt sonst rund 40 Migrationszeilen
    setup_mlflow("setup-check")
    logging.disable(logging.NOTSET)

    note = tmp / "setup_check.txt"
    note.write_text("Setup-Check der MLOps-Labs\n", encoding="utf-8")
    with mlflow.start_run(run_name="setup-check") as run:
        mlflow.log_param("model", "minilm-l6")
        mlflow.log_metric("texts_per_second", texts_per_second)
        mlflow.log_artifact(str(note))

    stored = mlflow.get_run(run.info.run_id)
    artifacts = [item.path for item in mlflow.MlflowClient().list_artifacts(run.info.run_id)]
    report(
        stored.data.params.get("model") == "minilm-l6"
        and "texts_per_second" in stored.data.metrics
        and "setup_check.txt" in artifacts,
        f"MLflow {mlflow.__version__}: Run mit Parameter, Metrik und Artefakt geschrieben und gelesen",
        "pip install -r requirements.txt",
    )
except Exception as exc:
    report(
        False,
        f"MLflow kann keinen Run schreiben ({type(exc).__name__}: {exc})",
        'pip install "mlflow>=2.22,<3" "sqlalchemy>=2.0,<2.1"',
    )
finally:
    logging.disable(logging.NOTSET)
    for key, value in saved_env.items():
        if value is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = value
    # unter Windows hält SQLite die Datei manchmal noch offen: dann bleibt der Ordner liegen
    shutil.rmtree(tmp, ignore_errors=True)

## Ergebnis

In [ ]:
if problems:
    print(f"{len(problems)} Mangel gefunden:" if len(problems) == 1 else f"{len(problems)} Mängel gefunden:")
    for number, problem in enumerate(problems, start=1):
        print(f"  {number}. {problem}")
else:
    print("Alles bereit")

Steht oben **Alles bereit**, können Sie mit Lab 1 "Embedding-Modelle evaluieren und reproduzierbar tracken"
beginnen. Andernfalls beheben Sie die Mängel in der genannten Reihenfolge und führen das Notebook erneut aus.